In [1]:
import numpy as np

In [3]:
import numpy as np

class Lasso:

    def __init__(self, alpha=1.0, epochs=1000, tol=1e-4):
        self.alpha = alpha       
        self.epochs = epochs    
        self.tol = tol           

        self.weights = None
        self.intercept = None

    def _soft_threshold(self, rho):
        if rho > self.alpha:
            return rho - self.alpha
        elif rho < -self.alpha:
            return rho + self.alpha
        else:
            return 0.0

    def fit(self, X, y):

        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).ravel()

        m, n = X.shape

        self.X_mean = X.mean(axis=0)
        self.y_mean = y.mean()
        X_centered = X - self.X_mean
        y_centered = y - self.y_mean

        self.weights = np.zeros(n)
        z = np.mean(X_centered ** 2, axis=0)

        for epoch in range(self.epochs):
            old_weights = self.weights.copy()
            for j in range(n):
                if z[j] == 0:
                    self.weights[j] = 0.0
                    continue
                residual = (y_centered - X_centered @ self.weights + X_centered[:, j] * self.weights[j])
                rho = np.mean(X_centered[:, j] * residual)

                self.weights[j] = (
                    self._soft_threshold(rho) / z[j]
                )
            change = np.max(np.abs(self.weights - old_weights))

            if change < self.tol:
                break

        self.intercept = (
            self.y_mean - self.X_mean @ self.weights
        )

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.weights + self.intercept